# Online-vehicle count & Redis storage sizing (fixed windows)

**This notebook does no computation of its own** — it only reads the 3 CSVs produced by
[`scripts/online_vehicle_redis_storage_analysis.py`](../scripts/online_vehicle_redis_storage_analysis.py),
which must be run **on the cloud instance** (real face embeddings only exist there:
`top_10k/embeddings/embeddings_{tenant_id}_{vehicle_id}.joblib`, empty in this local checkout).
Workflow: copy that script to the instance → run it there → copy
`system_design/outputs/*_online_storage_*.csv` back into this repo → run this notebook.

**Question**: for fixed clock-aligned windows of length `WL` (10/15/30/60 min), how many
vehicles are "online" (>=1 record arriving) in each window, how many records (1 image = 1
record) do they produce, and how much Redis storage would that require if every record cached
its own real embedding (measured from the instance's `.joblib` files, not assumed) plus 662
bytes of JSON metadata — a naive/worst-case "cost to hold everything raw until window close"
number, with no dis split-merge boundary-caching optimization applied and no ceiling clipping.


## Expected input schema

- **`vehicle_window_online_storage_results.csv`** — one row per
  `(tenant_id, vehicle_id, window_length_minutes, window_id, window_start)`:
  `record_count`, `storage_bytes`, `embeddings_missing` (records that fell back to the default
  4096-byte estimate because their `image_id` had no entry in the vehicle's embeddings file),
  `dropped_missing_timestamp`, `error` (non-null = this vehicle failed to load; all other fields
  null for that row).
- **`overall_window_online_storage_results.csv`** — one row per
  `(window_length_minutes, window_id, window_start)`: `n_online_vehicles` (nunique vehicles
  active in that window), `total_records`, `total_storage_bytes` (fleet-wide, summed across
  every online vehicle).
- **`online_storage_summary_by_window_length.csv`** — one row per `window_length_minutes`:
  vehicle/window counts, peak `n_online_vehicles` and peak `total_storage_bytes` (each with the
  `window_start` it occurred at), and mean/median of both across all windows of that length.


In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

OUTPUT_DIR = "../outputs"
VEHICLE_WINDOW_PATH = os.path.join(OUTPUT_DIR, "vehicle_window_online_storage_results.csv")
OVERALL_PATH = os.path.join(OUTPUT_DIR, "overall_window_online_storage_results.csv")
SUMMARY_PATH = os.path.join(OUTPUT_DIR, "online_storage_summary_by_window_length.csv")

WINDOW_LENGTHS_MINUTES = [10, 15, 30, 60]

HAS_DATA = all(os.path.exists(p) for p in (VEHICLE_WINDOW_PATH, OVERALL_PATH, SUMMARY_PATH))

if not HAS_DATA:
    print(
        "No data yet -- run scripts/online_vehicle_redis_storage_analysis.py on the cloud "
        "instance (real embeddings live there only) and copy its 3 output CSVs into "
        f"{os.path.abspath(OUTPUT_DIR)!r} before re-running this notebook.\n"
        "Expected files:\n"
        f"  {VEHICLE_WINDOW_PATH}\n  {OVERALL_PATH}\n  {SUMMARY_PATH}"
    )
else:
    vehicle_window_df = pd.read_csv(VEHICLE_WINDOW_PATH, parse_dates=["window_start"])
    overall_df = pd.read_csv(OVERALL_PATH, parse_dates=["window_start"])
    summary_df = pd.read_csv(SUMMARY_PATH, parse_dates=[
        "peak_n_online_vehicles_window_start", "peak_total_storage_bytes_window_start"
    ])
    print(f"Loaded: vehicle_window={vehicle_window_df.shape}, overall={overall_df.shape}, summary={summary_df.shape}")


No data yet -- run scripts/online_vehicle_redis_storage_analysis.py on the cloud instance (real embeddings live there only) and copy its 3 output CSVs into '/Users/swapnilmane/NetraDyne/DIS_logic/system_design/outputs' before re-running this notebook.
Expected files:
  ../outputs/vehicle_window_online_storage_results.csv
  ../outputs/overall_window_online_storage_results.csv
  ../outputs/online_storage_summary_by_window_length.csv


## Summary by window length

In [2]:
if HAS_DATA:
    display_df = summary_df.copy()
    for col in ["peak_total_storage_bytes", "mean_total_storage_bytes", "median_total_storage_bytes"]:
        display_df[col.replace("bytes", "gb")] = display_df[col] / (1024**3)
    display(display_df)
else:
    print("Skipped -- no data yet (see previous cell).")


Skipped -- no data yet (see previous cell).


## Online vehicles, records, and storage over time, per window length

In [3]:
if HAS_DATA:
    fig, axes = plt.subplots(len(WINDOW_LENGTHS_MINUTES), 3, figsize=(15, 3.2 * len(WINDOW_LENGTHS_MINUTES)), sharex=False)
    for row, wl in enumerate(WINDOW_LENGTHS_MINUTES):
        g = overall_df[overall_df["window_length_minutes"] == wl].sort_values("window_start")

        axes[row, 0].plot(g["window_start"], g["n_online_vehicles"], marker="o", markersize=3)
        axes[row, 0].set_title(f"WL={wl}min: online vehicles")
        axes[row, 0].set_ylabel("# vehicles")

        axes[row, 1].plot(g["window_start"], g["total_records"], marker="o", markersize=3)
        axes[row, 1].set_title(f"WL={wl}min: total records")
        axes[row, 1].set_ylabel("# records")

        axes[row, 2].plot(g["window_start"], g["total_storage_bytes"] / (1024**3), marker="o", markersize=3)
        axes[row, 2].set_title(f"WL={wl}min: total storage")
        axes[row, 2].set_ylabel("GB")

    fig.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, "online_storage_timeseries_by_wl.png")
    fig.savefig(plot_path, dpi=150)
    print(f"Wrote {plot_path}")
    plt.show()
else:
    print("Skipped -- no data yet (see first cell).")


Skipped -- no data yet (see first cell).


## Per-vehicle storage distribution, per window length

In [4]:
if HAS_DATA:
    ok = vehicle_window_df[vehicle_window_df["error"].isna()]
    fig, axes = plt.subplots(1, len(WINDOW_LENGTHS_MINUTES), figsize=(4 * len(WINDOW_LENGTHS_MINUTES), 3.5), sharey=True)
    for ax, wl in zip(axes, WINDOW_LENGTHS_MINUTES):
        vals = ok.loc[ok["window_length_minutes"] == wl, "storage_bytes"] / 1024  # KB
        ax.hist(vals.dropna(), bins=30)
        ax.set_title(f"WL = {wl} min")
        ax.set_xlabel("KB per vehicle-window")
    axes[0].set_ylabel("# (vehicle, window) rows")
    fig.suptitle("Per-vehicle-window storage_bytes distribution, by window length")
    fig.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, "online_storage_pervehicle_distribution_by_wl.png")
    fig.savefig(plot_path, dpi=150)
    print(f"Wrote {plot_path}")
    plt.show()
else:
    print("Skipped -- no data yet (see first cell).")


Skipped -- no data yet (see first cell).


## Peak fleet-wide storage vs. window length

In [5]:
if HAS_DATA:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(summary_df["window_length_minutes"], summary_df["peak_total_storage_bytes"] / (1024**3), marker="o")
    ax.set_xlabel("Window length (minutes)")
    ax.set_ylabel("Peak fleet-wide storage (GB)")
    ax.set_title("Peak Redis storage required vs. fixed window length")
    ax.set_xticks(WINDOW_LENGTHS_MINUTES)
    fig.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, "peak_online_storage_vs_wl.png")
    fig.savefig(plot_path, dpi=150)
    print(f"Wrote {plot_path}")
    plt.show()
else:
    print("Skipped -- no data yet (see first cell).")


Skipped -- no data yet (see first cell).


## Notes / caveats

- **Record granularity**: 1 image = 1 record — no `dis`/arrival-message grouping (contrast with
  `redis_load_analysis.py`'s arrival reconstruction elsewhere in this project).
- **Storage model**: 662 bytes JSON metadata (`incoming_requests_rate_analysis.ipynb`'s
  field-by-field derivation) + each record's **own real embedding size**, measured via `.nbytes`
  from the instance's `.joblib` files — not a hardcoded 4096. Falls back to 4096 only when an
  `image_id` has no embedding entry (`embeddings_missing`, tracked per vehicle-window).
- **No split-merge optimization**: every record caches its own embedding for as long as its
  window is open — this is the naive/worst-case "cost to hold everything" number, not what a
  boundary-caching design (method two) would actually need. Deliberately higher than that
  design's real footprint.
- **No upper bound**: raw computed bytes, not clipped against any Redis memory ceiling — for
  ceiling context (16GB / 80%-safe), see `fixed_window_and_window_timer_condition_method_one/analysis/plot_redis_memory.py`'s
  `RAW_CEILING_BYTES`/`SAFE_CEILING_BYTES` constants, not reproduced here.
- **"Peak per window" = held-until-close**: this raw model assumes every record from a window is
  still resident when that window closes (no incremental eviction/finalization mid-window).
